In [1]:
from appgeopy import *
from my_packages import *

Cannot find header.dxf (GDAL_DATA is not defined)


## export GPS timeseries to feather

## export timeseries MLCW and GPS side-by-side

### create incremental values of MLCW, GWL

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation


def remove_series_outliers(series, window=7, threshold=3):
    """Removes outliers from a pandas Series using a rolling Hampel filter.

    Returns a clean copy of the Series.
    """
    # 1. Calculate rolling statistics
    rolling_window = series.rolling(window, center=True)
    rolling_median = rolling_window.median()
    rolling_mad = rolling_window.apply(median_abs_deviation, raw=True)

    # 2. Identify outliers exceeding the threshold
    deviation = np.abs(series - rolling_median)
    outlier_mask = deviation > (threshold * rolling_mad)

    # 3. Replace outliers with the local rolling median
    cleaned_series = series.copy()
    cleaned_series[outlier_mask] = rolling_median[outlier_mask]

    return cleaned_series


# =================================================================
# EXAMPLE USAGE
# =================================================================
# target_series = ts_df["water_level"]
# ts_df["cleaned_level"] = remove_series_outliers(target_series, window=7, threshold=3)

#### incremental MLCW

#### incremental groundwater levels

#### incremental GPS

In [3]:
mlcw_diff_df = pd.read_feather(
    r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\tau_demo_TUKU\data\incremental_data\mlcw_diff_cleaned.feather"
)
mlcw_diff_df = mlcw_diff_df.set_index("datetime")
mlcw_diff_df.head(5)

,F1,T1,F2,T2,F3,F4
datetime,,,,,,
2003-12-06,NaN,NaN,NaN,NaN,NaN,NaN
2003-12-11,0.011516,-0.055645,0.080997,0.087460,0.151367,0.009768
2003-12-16,0.019792,-0.061439,0.053173,0.072521,0.151376,0.011909
2003-12-21,0.026245,-0.067164,0.017787,0.054084,0.148342,0.014116
2003-12-26,0.030629,-0.072749,-0.024748,0.032609,0.142126,0.016282


In [12]:
gps_fpath = r"D:\1000_SCRIPTS\004_Project003\20260427_InSAR_MLCW_v2\tau_demo_TUKU\data\TUKU_GPS_timeseries.feather"
gps_df = pd.read_feather(gps_fpath)
gps_df = gps_df.set_index("date")
mutual_index = gps_df.index.intersection(mlcw_diff_df.index)
gps_df = gps_df.loc[mutual_index, :]

gps_diff_df = gps_df.copy()

temp = gps_df["modeled"]
temp_diff = temp.diff()
temp_diff_cleaned = remove_series_outliers(temp_diff, window=7, threshold=3)
gps_diff_df["modeled"] = temp_diff_cleaned
gps_diff_df = gps_diff_df.reset_index()
gps_diff_df = gps_diff_df.rename({"index": "datetime"}, axis=1)
gps_diff_df.to_feather(gps_fpath.replace("GPS", "GPS_diff"))
gps_diff_df.head(5)

,datetime,modeled
0,2010-01-06,NaN
1,2010-01-11,-0.283296
2,2010-01-16,-0.447508
3,2010-01-21,-0.619063
4,2010-01-26,-0.793238
